# Módulo de Configuración Histórica
## Notebook 01 — Catálogos / Tablas de Dimensiones

Este notebook crea las cuatro tablas estáticas que actúan como catálogos del módulo:

| Tabla | Descripción |
|---|---|
| `dim_usuario` | Quién — personas registradas en el sistema |
| `dim_rol` | Qué papel juega cada persona |
| `dim_evaluacion` | Qué se mide (métricas e indicadores) |
| `dim_periodo` | Cuándo — catálogo de meses/períodos |

> **Prerrequisito**: Asegúrate de que este notebook esté adjunto a un Lakehouse antes de ejecutarlo.  
> En la barra lateral de Fabric selecciona **Add Lakehouse** y elige tu Lakehouse de destino.

---
## 0 · Configuración del Lakehouse

Define el nombre del Lakehouse que usarás. Todas las tablas se crearán dentro de él.

In [ ]:
# ─── Ajusta este valor al nombre de tu Lakehouse en Fabric ───────────────────
LAKEHOUSE_NAME = "BI - Bandelta LH"
# ─────────────────────────────────────────────────────────────────────────────

spark.sql(f"USE `{LAKEHOUSE_NAME}`")
print(f"✅ Usando Lakehouse: {LAKEHOUSE_NAME}")

---
## 1 · dim_usuario

Catálogo de personas que participan en el modelo de compensación.

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS dim_usuario (
    id_usuario      INT           NOT NULL  COMMENT 'Llave primaria del usuario',
    nombre          STRING        NOT NULL  COMMENT 'Nombre completo',
    email           STRING        NOT NULL  COMMENT 'Correo electrónico corporativo (único)',
    codigo_empleado STRING                  COMMENT 'Número de empleado en el sistema RH',
    area            STRING                  COMMENT 'Área o departamento al que pertenece',
    activo          BOOLEAN       NOT NULL  COMMENT 'Indica si el usuario está activo',
    fecha_creacion  TIMESTAMP     NOT NULL  COMMENT 'Timestamp de alta en el sistema',
    fecha_baja      TIMESTAMP               COMMENT 'Timestamp de baja; NULL si sigue activo'
)
USING DELTA
COMMENT 'Catálogo de usuarios del módulo de compensación histórica'
TBLPROPERTIES (
    'delta.minReaderVersion' = '1',
    'delta.minWriterVersion' = '2'
)
""")

print("✅ Tabla dim_usuario creada (o ya existía).")

In [ ]:
# Verificar estructura
spark.sql("DESCRIBE TABLE dim_usuario").show(truncate=False)

---
## 2 · dim_rol

Catálogo de roles o posiciones dentro del modelo comercial.

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS dim_rol (
    id_rol       INT     NOT NULL  COMMENT 'Llave primaria del rol',
    nombre_rol   STRING  NOT NULL  COMMENT 'Nombre del rol, p.ej. Asesor Comercial',
    descripcion  STRING            COMMENT 'Descripción detallada del rol',
    nivel        STRING            COMMENT 'Nivel jerárquico: Operativo, Táctico, Estratégico',
    activo       BOOLEAN NOT NULL  COMMENT 'Indica si el rol está vigente'
)
USING DELTA
COMMENT 'Catálogo de roles del modelo de compensación'
TBLPROPERTIES (
    'delta.minReaderVersion' = '1',
    'delta.minWriterVersion' = '2'
)
""")

print("✅ Tabla dim_rol creada (o ya existía).")

In [ ]:
spark.sql("DESCRIBE TABLE dim_rol").show(truncate=False)

---
## 3 · dim_evaluacion

Catálogo de indicadores o métricas que se miden en los esquemas de compensación.

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS dim_evaluacion (
    id_evaluacion      INT     NOT NULL  COMMENT 'Llave primaria de la evaluación',
    nombre_evaluacion  STRING  NOT NULL  COMMENT 'Nombre del indicador, p.ej. Ventas Netas',
    descripcion        STRING            COMMENT 'Qué mide y cómo se calcula',
    tipo_metrica       STRING  NOT NULL  COMMENT 'porcentaje | valor_absoluto | ratio | conteo',
    unidad_medida      STRING            COMMENT 'MXN, unidades, %, clientes, etc.',
    activo             BOOLEAN NOT NULL  COMMENT 'Indica si la evaluación está vigente'
)
USING DELTA
COMMENT 'Catálogo de evaluaciones/indicadores del modelo de compensación'
TBLPROPERTIES (
    'delta.minReaderVersion' = '1',
    'delta.minWriterVersion' = '2'
)
""")

print("✅ Tabla dim_evaluacion creada (o ya existía).")

In [ ]:
spark.sql("DESCRIBE TABLE dim_evaluacion").show(truncate=False)

---
## 4 · dim_periodo

Catálogo de períodos (generalmente meses). Es la dimensión temporal del modelo.

> **Tip**: Pre-genera los períodos de varios años para que las tablas de metas puedan referenciarlos sin necesidad de inserción en línea.

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS dim_periodo (
    id_periodo     INT     NOT NULL  COMMENT 'Llave primaria del período',
    anio           INT     NOT NULL  COMMENT 'Año, p.ej. 2024',
    mes            INT     NOT NULL  COMMENT 'Mes numérico 1-12',
    nombre_periodo STRING  NOT NULL  COMMENT 'Etiqueta legible, p.ej. Enero 2024',
    fecha_inicio   DATE    NOT NULL  COMMENT 'Primer día del período',
    fecha_fin      DATE    NOT NULL  COMMENT 'Último día del período'
)
USING DELTA
COMMENT 'Catálogo de períodos (meses) para el modelo de compensación'
TBLPROPERTIES (
    'delta.minReaderVersion' = '1',
    'delta.minWriterVersion' = '2'
)
""")

print("✅ Tabla dim_periodo creada (o ya existía).")

In [ ]:
# ─── Generación automática de períodos 2023-2026 ──────────────────────────────
from pyspark.sql import Row
import calendar
from datetime import date

MESES_ES = [
    "Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio",
    "Julio", "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre"
]

periodos = []
pid = 1
for anio in range(2023, 2027):          # Ajusta el rango según tus necesidades
    for mes in range(1, 13):
        ultimo_dia = calendar.monthrange(anio, mes)[1]
        periodos.append(Row(
            id_periodo     = pid,
            anio           = anio,
            mes            = mes,
            nombre_periodo = f"{MESES_ES[mes-1]} {anio}",
            fecha_inicio   = date(anio, mes, 1),
            fecha_fin      = date(anio, mes, ultimo_dia)
        ))
        pid += 1

df_periodos = spark.createDataFrame(periodos)

# Solo inserta si la tabla está vacía (evita duplicados en re-ejecuciones)
existing = spark.sql("SELECT COUNT(*) as cnt FROM dim_periodo").collect()[0].cnt
if existing == 0:
    df_periodos.write.mode("append").saveAsTable("dim_periodo")
    print(f"✅ {len(periodos)} períodos insertados en dim_periodo.")
else:
    print(f"ℹ️  dim_periodo ya contiene {existing} registros. No se insertaron duplicados.")

spark.sql("SELECT * FROM dim_periodo ORDER BY id_periodo").show(50, truncate=False)

---
## 5 · Resumen de tablas creadas

In [ ]:
tablas_dim = ["dim_usuario", "dim_rol", "dim_evaluacion", "dim_periodo"]

print(f"{'TABLA':<30} {'REGISTROS':>10}")
print("-" * 42)
for tabla in tablas_dim:
    cnt = spark.sql(f"SELECT COUNT(*) as c FROM {tabla}").collect()[0].c
    print(f"{tabla:<30} {cnt:>10,}")

print("\n✅ Notebook 01 completado. Continúa con 02_asignaciones_y_esquemas.ipynb")